# TB Portals — **Agentic 6-Rung Pipeline (headline run)**: mode `a2`
Caches RAD-DINO CLS+patch-grid features, runs rungs `1 2 3 5 6` + agentic_best variants + spatial-cavity probe, **5 seeds × M=10**, saves heads. Download `agentic_a2.zip`.
Attach only **tb-portals-cxr-pngs**; Internet **ON**; GPU T4.

## 0 — Clone  *(restart kernel after any pull that changed .py)*

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + "/scripts"):
    if _p not in sys.path: sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)

## Install deps

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "torchxrayvision", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")

## Paths

In [ ]:
import os
WORK          = "/kaggle/working"
REPO_DIR      = "/kaggle/working/dl-project-codebase"
DATASET       = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT = f"{DATASET}/kaggle_export"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
MODE          = "a2"
BACKBONE      = "rad-dino"
FEATURES      = f"{WORK}/features_{BACKBONE}_cls.npz"
FEATURES_GRID = f"{WORK}/features_{BACKBONE}_grid7.npz"
print("aux grid features ->", FEATURES_GRID)
print("MODE:", MODE, "| KAGGLE_EXPORT:", os.path.isdir(KAGGLE_EXPORT), "| primary features ->", FEATURES)

## 1 — Build the 5,010-image manifest (Kantipudi Table 1)

In [ ]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

## 2 — Cache primary features (one-time; idempotent)

In [ ]:
import os, sys
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
if os.path.isfile(FEATURES):
    print("primary features already cached ->", FEATURES)
else:
    from cache_features import main as cache_main
    cache_main(["--manifest", PAPER_MANIFEST, "--out", FEATURES,
                "--backbone", BACKBONE, "--batch-size", "32"])

## 2b — Cache patch-grid features (for the spatial cavity head)

In [ ]:
if os.path.isfile(FEATURES_GRID):
    print("grid features already cached ->", FEATURES_GRID)
else:
    from cache_features import main as cache_main
    cache_main(["--manifest", PAPER_MANIFEST, "--out", FEATURES_GRID,
                "--backbone", BACKBONE, "--patch-grid", "7", "--batch-size", "32"])

## 3 — Run the rung ladder for mode `a2` (rungs 1 2 3 5 6, 5 seeds, M=10)

In [ ]:
from src.training.train_agentic import main as agentic_main
args = ["--features", FEATURES, "--manifest", PAPER_MANIFEST,
        "--mode", MODE, "--out-dir", f"{WORK}/agentic_{MODE}",
        "--rungs", '1', '2', '3', '5', '6',
        "--seeds", '0', '1', '2', '3', '4',
        "--ensemble-m", "10",
        "--held-outs", "Romania", "Moldova", "Kazakhstan",
        "--save-heads"]
args += ["--features-grid", FEATURES_GRID]
agentic_main(args)

## 4 — Save (download → baseline_runs/agentic_runs/)

In [ ]:
import os, shutil
src = f"{WORK}/agentic_{MODE}"
try: shutil.copy(FEATURES, f"{src}/{os.path.basename(FEATURES)}")
except Exception as e: print("primary cache copy skipped:", e)
try: shutil.copy(FEATURES_GRID, f"{src}/{os.path.basename(FEATURES_GRID)}")
except Exception as e: print("grid cache copy skipped:", e)
zip_path = shutil.make_archive(f"{WORK}/agentic_{MODE}", "zip", src)
print("Saved ->", zip_path)
print("Contents include: results_agentic_*.csv, preds_*.csv, heads/*.pt, features_*.npz")